# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abc085455-byte/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring** (a continuous per-page priority score, built on top of a **binary classification** sub-task).

I'm not just sorting pages by one raw number (that would be ranking on a fixed metric) and I'm not grouping look-alike pages with no label (that would be clustering). I need a **score per page** — `is this page worth reviewing this week?` — that a classifier can estimate as a probability, which I then use to order the whole queue. That's exactly the shape the starter pipeline uses: `final_score = 0.70 * model_probability + 0.30 * normalized_baseline_score`, then the queue is the pages sorted by that score. So underneath, it's binary classification (declining vs. not); on top, it's a scoring/ranking output because the *action* (review top N) only needs an ordered list, not hard labels.


In [ ]:
# Quick sanity note, not a heavy computation for this section:
# the starter pipeline (outputs/model_report.md, already run and committed in this repo)
# confirms this is framed as classification-under-the-hood -> score-on-top:
with open("../../outputs/model_report.md") as f:
    report = f.read()
print(report.split("## Model Comparison")[0])


# FlyRank Refresh Opportunity Model Report

This report is generated from the bundled anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`).
The model ranks existing content for refresh review. It does not use titles, URLs, client names, domains, or keywords.

## Data

- Rows scored: 30,000
- Declining-label rows: 16,262
- Declining-label rate: 0.542
- Split strategy used for validation: client_holdout
- Target: `is_declining_label`




## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (starter proxy): `is_declining_label = (trend_direction == "down")`.**

This label comes from a **defined rule on the current window**, not a future observed outcome — the lane guide is explicit about this being a beginner proxy, not the ideal capstone target. `trend_direction` is already a summary of recent movement (see `trend_pct` below), so today I'm predicting "is this page *currently* trending down," not "will this page decline *next*."

For my actual capstone (later weeks) I want to move to a real **future-window label**: something like *"did this page's impressions drop by more than X% in the 30 days AFTER a cutoff date, given its behavior in the 30 days before it"* — a proper `prior feature window -> future target window` split with a client-holdout, so I'm not leaking today's answer into today's features. For this week's framing exercise, I'm using the current-window proxy because it lets me sketch the whole loop honestly and cheaply, and I'm naming its weakness instead of hiding it.


In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down")

print("declining-label rate:", round(df["is_declining_label"].mean(), 3))
print(df["trend_direction"].value_counts())


declining-label rate: 0.542
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50** — of the top 50 pages my score puts at the front of the queue, how many are actually positive by `is_declining_label`?

I'm choosing this over plain accuracy or ROC AUC because the *action* is "an editor manually reviews a short, capacity-limited list" — nobody reviews all 30,000 pages, so what matters is whether the top of the queue is trustworthy, not the whole ranking. This is already measured and committed in this repo's own `outputs/model_report.md`: the hand-written baseline rule scores **0.240** precision@50 (about 12 of its top 50 are real), while the random forest scores **0.740** (about 37 of 50) — a large, real gap on this exact dataset.

One honest caveat from Week 1's cost analysis: a missed decliner (false negative) is more expensive than a wasted review (false positive), so I'll also keep an eye on **recall** as a secondary check — the report shows the random forest's recall (0.744) is close to its precision, which is a good early sign it isn't just being precision-greedy at recall's expense.


In [ ]:
import pandas as pd

report_table = """
| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |
"""
print(report_table)
print("Baseline catches about", round(0.240 * 50), "of its top 50 correctly.")
print("Random forest catches about", round(0.740 * 50), "of its top 50 correctly.")



| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

Baseline catches about 12 of its top 50 correctly.
Random forest catches about 37 of its top 50 correctly.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page, for one client, as of this data snapshot** (`content_id` x `client_id`). It is *not* one row per day (that's the much larger daily fact table) and *not* one row per client — each client has many pages, and each page here already carries 90-day rolled-up performance numbers (impressions, clicks, CTR, trend) plus content metadata (age, word count, freshness). That rolled-up-per-page shape is exactly the grain my scoring task needs, since the action ("review this page") also happens at the page level.


In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down")

cols = [
    "content_id", "client_id", "content_age_days", "days_since_last_update",
    "impressions_90d", "ctr", "avg_position", "trend_direction", "trend_pct",
    "is_declining_label",
]
print("rows:", len(df), "| one row = one content_id x client_id")
df[cols].head(5)


rows: 30000 | one row = one content_id x client_id


,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,187,20,3803,0.76,10.6,down,-41.4,True
1,content_a1fb4e703a9e,client_4e07408562,445,25,15320,0.05,20.3,down,-57.7,True
2,content_9aa793d4d895,client_7f2253d7e2,141,20,12581,0.09,36.5,down,-60.9,True
3,content_331d6c4de07b,client_19581e27de,463,22,11751,0.49,6.2,stable,-13.8,False
4,content_d99b7a2d90ca,client_3fdba35f04,263,14,19140,0.13,44.0,down,-34.7,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The starter **baseline is literally a set of if-statements**: `days_since_last_update >= 180 AND impressions_90d >= 500` -> stale, `trend_direction == "down" AND impressions_90d >= 100` -> declining, and so on — each rule looks at one or two columns with a fixed cutoff, in isolation. Real pages don't fail for one clean reason: a page can be borderline-stale, medium-traffic, slightly-declining, and mid-position all at once, and none of those alone crosses a rule's threshold, even though *together* they're a strong signal. A fixed rule can't weigh 15+ numeric signals against each other or learn that, say, `days_since_last_update` matters a lot more when `impressions_90d` is also high but matters little when a page has almost no traffic anyway.

That's borne out in the numbers already committed in this repo: the rule-based baseline gets **ROC AUC 0.627 / precision@50 0.240**, while a random forest trained on the same features gets **ROC AUC 0.750 / precision@50 0.740** — nearly triple the top-50 precision, on the *same* underlying data, just by letting a model learn the interactions instead of hand-picking thresholds. The model's own top features (`days_with_impressions`, `log_impressions_90d`, `avg_position`, per `outputs/model_report.md`) aren't wildly different from what the baseline already looks at — the gain comes from combining them, which is exactly what an if-statement can't do well.


In [ ]:
with open("../../outputs/model_report.md") as f:
    report = f.read()
print(report.split("## Top Features")[1][:400])




- `days_with_impressions`: 0.1578
- `log_impressions_90d`: 0.1282
- `avg_position`: 0.1090
- `content_age_days`: 0.0955
- `char_count`: 0.0426
- `word_count`: 0.0397
- `log_clicks_90d`: 0.0346
- `ctr`: 0.0330
- `scroll_rate`: 0.0311
- `days_with_sessions`: 0.0280

## Top 10 Queue Preview

| Rank | Score | Model probability | Action | Reasons | Impressions | Sessions | Trend |
|---:|---:|---:|---


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.